
# 20.21 — Cold-start vs continuation
## Separar dependência do otimizador, multimodalidade do landscape e limitação do ansatz

Este experimento é um **controle causal** da 20.20. A fronteira clássica \(s_c\) já foi identificada por enumeração; agora a pergunta é:

> As falhas/alterações observadas perto da fronteira vêm da continuação por *warm start*, de multimodalidade do landscape, ou de uma limitação real de expressividade do ansatz?

Para cada ponto da campanha fina, serão comparados três protocolos:

1. **forward continuation** — resultado já produzido na 20.20;
2. **backward continuation** — resultado já produzido na 20.20;
3. **cold starts independentes** — \(N\) inicializações independentes no mesmo Hamiltoniano \(H(s)\).

### Critério científico
- `forward == backward == cold` → regime robusto;
- `forward != backward`, mas cold encontra múltiplas bacias → dependência de caminho / landscape multimodal;
- continuação falha, mas cold recupera o ótimo → aprisionamento induzido por *warm start*;
- todos os protocolos falham → **falha não resolvida**; só chamar de candidato a limitação do ansatz se uma auditoria independente de reachability/expressividade também falhar.

**Não usar \(s_c\), \(\delta=s-s_c\), ótimo clássico ou qualquer diagnóstico pós-solução como INPUT de Transformer.**


In [ ]:

from __future__ import annotations

import json
import math
import inspect
import hashlib
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

SEED = 20260828
N_COLD_STARTS = 10

P_OPT_SUCCESS = 0.90
P_OPT_STRONG = 0.99
ENERGY_TOL = 1e-6
REACHABILITY_TOL = 1e-6

rng = np.random.default_rng(SEED)

# Diretórios: reaproveita a convenção da 20.20 se estiverem no kernel.
ROOT = Path.cwd()

if "VALIDATION20_TABLE_DIR" in globals():
    TABLE_DIR = Path(VALIDATION20_TABLE_DIR)
else:
    TABLE_DIR = ROOT / "validation_20_21" / "tables"

if "VALIDATION20_FIGURE_DIR" in globals():
    FIGURE_DIR = Path(VALIDATION20_FIGURE_DIR)
else:
    FIGURE_DIR = ROOT / "validation_20_21" / "figures"

CHECKPOINT_DIR = ROOT / "validation_20_21" / "checkpoints"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("TABLE_DIR     =", TABLE_DIR.resolve())
print("FIGURE_DIR    =", FIGURE_DIR.resolve())
print("CHECKPOINT_DIR=", CHECKPOINT_DIR.resolve())



## Célula 3 — localizar automaticamente a campanha fina da 20.20

O notebook procura primeiro `DataFrame`s já existentes no kernel. Se não encontrar, procura CSVs com as colunas mínimas:

- `k`
- `asset_index`
- `return_shock_sigma_units`
- `delta_from_boundary`
- `critical_return_shock_sigma_units`

Isso evita depender do nome exato dado ao objeto na 20.20.


In [ ]:

REQUIRED_FINE_COLUMNS = {
    "k",
    "asset_index",
    "return_shock_sigma_units",
    "delta_from_boundary",
    "critical_return_shock_sigma_units",
}

def _candidate_dataframes():
    out = []
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            cols = set(obj.columns)
            if REQUIRED_FINE_COLUMNS.issubset(cols):
                out.append((name, obj.copy()))
    return out

def _candidate_csvs():
    roots = []
    if "VALIDATION20_TABLE_DIR" in globals():
        roots.append(Path(VALIDATION20_TABLE_DIR))
    roots += [ROOT, ROOT / "validation_20_20", ROOT / "validation_20_20" / "tables",
              ROOT / "validation20", ROOT / "tables"]
    seen = set()
    found = []
    for base in roots:
        if not base.exists():
            continue
        for p in base.rglob("*.csv"):
            if p in seen:
                continue
            seen.add(p)
            try:
                df = pd.read_csv(p, nrows=5)
            except Exception:
                continue
            if REQUIRED_FINE_COLUMNS.issubset(df.columns):
                found.append(p)
    return found

df_candidates = _candidate_dataframes()
csv_candidates = _candidate_csvs()

if df_candidates:
    # Preferência por nomes relacionados a fine/boundary.
    df_candidates.sort(key=lambda x: (("fine" not in x[0].lower()), ("boundary" not in x[0].lower()), x[0]))
    SOURCE_NAME, fine_20_20_df = df_candidates[0]
    print("Usando DataFrame do kernel:", SOURCE_NAME)
elif csv_candidates:
    csv_candidates.sort(key=lambda p: (("fine" not in p.name.lower()), ("boundary" not in p.name.lower()), len(str(p))))
    src = csv_candidates[0]
    SOURCE_NAME = str(src)
    fine_20_20_df = pd.read_csv(src)
    print("Usando CSV:", src)
else:
    raise RuntimeError(
        "Não encontrei a campanha fina da 20.20. "
        "Execute a 20.20 no mesmo kernel OU defina manualmente `fine_20_20_df` "
        "com as colunas mínimas exigidas."
    )

# Mantém somente a grade fina; remove duplicatas por cenário/direção quando possível.
fine_20_20_df = fine_20_20_df.copy()

sort_cols = [c for c in ["k", "direction", "delta_from_boundary", "asset_index"] if c in fine_20_20_df.columns]
if sort_cols:
    fine_20_20_df = fine_20_20_df.sort_values(sort_cols).reset_index(drop=True)

print("shape =", fine_20_20_df.shape)
display(fine_20_20_df.head(12))



## Célula 4 — contrato do solver

A 20.21 precisa chamar **o mesmo solver/VQE da 20.20**, alterando apenas a inicialização.

O notebook tenta localizar automaticamente uma função de execução já existente no kernel. Caso sua função tenha outro nome, basta fazer:

```python
SCENARIO_RUNNER = minha_funcao_da_20_20
```

### Saída esperada
O runner pode retornar `dict`, `Series` ou uma linha de `DataFrame`. Quanto mais campos abaixo ele fornecer, melhor:

`p_optimal`, `expected_energy`, `dominant_bitstring`, `qgt_numeric_rank`,
`qgt_rank_fraction`, `qgt_participation_dimension`, `active_parameter_fraction`,
`dominant_route_replacement_length`, `dominant_route_stretch`, `target_reachability`.

O código **não inventa** métricas ausentes.


In [ ]:

RUNNER_CANDIDATES = [
    "SCENARIO_RUNNER",
    "run_scenario_20_20",
    "solve_scenario_20_20",
    "solve_instance_20_20",
    "run_vqe_scenario",
    "solve_portfolio_vqe",
    "run_single_scenario",
    "solve_single_scenario",
]

def resolve_runner():
    for name in RUNNER_CANDIDATES:
        obj = globals().get(name)
        if callable(obj):
            return name, obj
    return None, None

RUNNER_NAME, SCENARIO_RUNNER = resolve_runner()

if SCENARIO_RUNNER is None:
    print("AUTO-ADAPTER: nenhum runner reconhecido no kernel.")
    print("Defina `SCENARIO_RUNNER = sua_funcao_da_20_20` antes da Célula 7.")
else:
    print("Runner detectado:", RUNNER_NAME)
    try:
        print("Assinatura:", inspect.signature(SCENARIO_RUNNER))
    except Exception:
        pass



## Célula 5 — normalização da saída e adapter de chamada

A função abaixo tenta assinaturas comuns sem assumir que a sua implementação tenha nomes específicos.
Se nenhuma assinatura casar, ela falha explicitamente em vez de mascarar o problema.


In [ ]:

STANDARD_ALIASES = {
    "energy": "expected_energy",
    "final_energy": "expected_energy",
    "vqe_energy": "expected_energy",
    "prob_optimal": "p_optimal",
    "optimal_probability": "p_optimal",
    "dominant_state": "dominant_bitstring",
    "dominant_bit_string": "dominant_bitstring",
    "qgt_rank": "qgt_numeric_rank",
    "participation_dimension": "qgt_participation_dimension",
    "active_fraction": "active_parameter_fraction",
    "route_stretch": "dominant_route_stretch",
    "route_replacement_length": "dominant_route_replacement_length",
    "reachability": "target_reachability",
}

def normalize_result(result):
    if result is None:
        return {}
    if isinstance(result, pd.DataFrame):
        if len(result) != 1:
            raise ValueError("O runner retornou DataFrame com mais de uma linha.")
        d = result.iloc[0].to_dict()
    elif isinstance(result, pd.Series):
        d = result.to_dict()
    elif isinstance(result, dict):
        d = dict(result)
    else:
        # Tenta dataclass/objeto simples.
        if hasattr(result, "__dict__"):
            d = dict(vars(result))
        else:
            raise TypeError(f"Saída do runner não suportada: {type(result)!r}")

    for old, new in STANDARD_ALIASES.items():
        if old in d and new not in d:
            d[new] = d[old]
    return d

def call_runner(row: pd.Series, seed: int, init_mode: str = "cold"):
    if SCENARIO_RUNNER is None:
        raise RuntimeError(
            "SCENARIO_RUNNER não definido. "
            "Atribua aqui a MESMA função usada pela 20.20 para resolver um cenário."
        )

    scenario = row.to_dict()

    attempts = [
        lambda: SCENARIO_RUNNER(row=row, seed=seed, init_mode=init_mode, warm_start=None),
        lambda: SCENARIO_RUNNER(scenario=row, seed=seed, init_mode=init_mode, warm_start=None),
        lambda: SCENARIO_RUNNER(scenario=scenario, seed=seed, init_mode=init_mode, warm_start=None),
        lambda: SCENARIO_RUNNER(row=row, seed=seed, cold_start=True),
        lambda: SCENARIO_RUNNER(scenario=scenario, seed=seed, cold_start=True),
        lambda: SCENARIO_RUNNER(scenario, seed=seed, init_mode=init_mode),
        lambda: SCENARIO_RUNNER(scenario, seed=seed),
    ]

    errors = []
    for i, fn in enumerate(attempts):
        try:
            return normalize_result(fn())
        except TypeError as e:
            errors.append(f"tentativa {i+1}: {e}")
            continue

    raise TypeError(
        "Não consegui adaptar automaticamente a assinatura do solver.\n"
        "Crie uma função wrapper com assinatura:\n\n"
        "    def SCENARIO_RUNNER(scenario, seed, init_mode='cold', warm_start=None):\n"
        "        ...\n"
        "        return {'p_optimal': ..., 'expected_energy': ..., ...}\n\n"
        + "\n".join(errors[-4:])
    )



## Célula 6 — escolher pontos físicos únicos

Forward e backward representam o mesmo \(H(s)\) quando `k`, `asset_index`, `return_shock_sigma_units`
e `delta_from_boundary` coincidem. Para os cold starts, cada Hamiltoniano deve ser resolvido **uma única vez por seed**.


In [ ]:

PHYSICAL_KEY = [c for c in [
    "k",
    "asset_index",
    "return_shock_sigma_units",
    "risk_scale",
    "critical_return_shock_sigma_units",
    "delta_from_boundary",
] if c in fine_20_20_df.columns]

physical_points_df = (
    fine_20_20_df
    .sort_values(PHYSICAL_KEY)
    .drop_duplicates(PHYSICAL_KEY)
    .reset_index(drop=True)
)

def stable_uid(row):
    payload = "|".join(f"{c}={row.get(c)}" for c in PHYSICAL_KEY)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

if "scenario_uid" not in physical_points_df.columns:
    physical_points_df["scenario_uid"] = physical_points_df.apply(stable_uid, axis=1)

print("Pontos físicos únicos:", len(physical_points_df))
display(physical_points_df[[c for c in ["scenario_uid"] + PHYSICAL_KEY if c in physical_points_df.columns]].head(15))



## Célula 7 — campanha cold-start com checkpoint

- \(N_{\rm cold}=10\) por padrão;
- cada seed é independente;
- resultados são salvos a cada execução;
- uma execução interrompida pode continuar sem repetir o que já foi calculado.


In [ ]:

COLD_CHECKPOINT = CHECKPOINT_DIR / "cold_start_runs.csv"

if COLD_CHECKPOINT.exists():
    cold_runs = pd.read_csv(COLD_CHECKPOINT)
    print("Checkpoint encontrado:", len(cold_runs), "execuções")
else:
    cold_runs = pd.DataFrame()

done = set()
if not cold_runs.empty and {"scenario_uid", "cold_seed"}.issubset(cold_runs.columns):
    done = set(zip(cold_runs["scenario_uid"].astype(str), cold_runs["cold_seed"].astype(int)))

rows_out = [] if cold_runs.empty else cold_runs.to_dict("records")
total = len(physical_points_df) * N_COLD_STARTS
counter = len(done)

for i, row in physical_points_df.iterrows():
    uid = str(row["scenario_uid"])
    for j in range(N_COLD_STARTS):
        cold_seed = int(SEED + 100_000 * i + j)
        key = (uid, cold_seed)
        if key in done:
            continue

        counter += 1
        print(
            f"[20.21 cold] {counter}/{total} "
            f"k={row.get('k')} delta={row.get('delta_from_boundary'):+.4f} seed={cold_seed}"
        )

        result = call_runner(row, seed=cold_seed, init_mode="cold")

        rec = {
            "scenario_uid": uid,
            "cold_seed": cold_seed,
            **{c: row.get(c) for c in physical_points_df.columns if c != "scenario_uid"},
            **result,
        }

        if "exact_energy" not in rec and "exact_energy" in row:
            rec["exact_energy"] = row["exact_energy"]

        if "expected_energy" in rec and "exact_energy" in rec:
            try:
                rec["energy_gap_abs"] = abs(float(rec["expected_energy"]) - float(rec["exact_energy"]))
            except Exception:
                pass

        rows_out.append(rec)
        pd.DataFrame(rows_out).to_csv(COLD_CHECKPOINT, index=False)

cold_runs = pd.DataFrame(rows_out)
print("Campanha cold-start concluída:", cold_runs.shape)
display(cold_runs.head())



## Célula 8 — agregação estatística dos cold starts

O sucesso não é definido apenas por energia. Quando disponível, usa-se:

\[
P_{\rm opt}\ge 0.9
\]

e também se registra a tolerância de energia.

Além disso medimos:
- melhor/mediana de \(P_{\rm opt}\);
- taxa de sucesso;
- número de estados dominantes distintos;
- entropia da distribuição dos estados dominantes;
- melhor e mediana de \(|E-E_0|\).


In [ ]:

def shannon_entropy_from_labels(s):
    vc = pd.Series(s).dropna().astype(str).value_counts(normalize=True)
    if len(vc) == 0:
        return np.nan
    return float(-(vc * np.log2(vc)).sum())

def bool_success(row):
    checks = []
    if pd.notna(row.get("p_optimal", np.nan)):
        checks.append(float(row["p_optimal"]) >= P_OPT_SUCCESS)
    if pd.notna(row.get("energy_gap_abs", np.nan)):
        checks.append(float(row["energy_gap_abs"]) <= ENERGY_TOL)
    # Se ambas existem, exigir ambas. Se só uma existe, usar a disponível.
    return bool(all(checks)) if checks else np.nan

cold_runs["cold_success"] = cold_runs.apply(bool_success, axis=1)

agg_spec = {
    "n_cold": ("cold_seed", "count"),
    "cold_success_fraction": ("cold_success", "mean"),
}

if "p_optimal" in cold_runs.columns:
    agg_spec.update({
        "cold_p_optimal_best": ("p_optimal", "max"),
        "cold_p_optimal_median": ("p_optimal", "median"),
        "cold_p_optimal_min": ("p_optimal", "min"),
    })

if "energy_gap_abs" in cold_runs.columns:
    agg_spec.update({
        "cold_energy_gap_best": ("energy_gap_abs", "min"),
        "cold_energy_gap_median": ("energy_gap_abs", "median"),
    })

if "dominant_bitstring" in cold_runs.columns:
    agg_spec.update({
        "cold_unique_dominant_states": ("dominant_bitstring", pd.Series.nunique),
        "cold_state_entropy_bits": ("dominant_bitstring", shannon_entropy_from_labels),
    })

if "qgt_rank_fraction" in cold_runs.columns:
    agg_spec.update({
        "cold_qgt_rank_fraction_median": ("qgt_rank_fraction", "median"),
        "cold_qgt_rank_fraction_std": ("qgt_rank_fraction", "std"),
    })

if "qgt_participation_dimension" in cold_runs.columns:
    agg_spec.update({
        "cold_qgt_participation_median": ("qgt_participation_dimension", "median"),
        "cold_qgt_participation_std": ("qgt_participation_dimension", "std"),
    })

cold_summary = (
    cold_runs
    .groupby("scenario_uid", as_index=False)
    .agg(**agg_spec)
)

meta_cols = [c for c in ["scenario_uid"] + PHYSICAL_KEY + ["exact_energy", "exact_bitstrings_asset_order"] if c in physical_points_df.columns]
cold_summary = physical_points_df[meta_cols].merge(cold_summary, on="scenario_uid", how="left")

cold_summary.to_csv(TABLE_DIR / "cold_start_summary_20_21.csv", index=False)
display(cold_summary.head(20))



## Célula 9 — reconstruir forward/backward da 20.20

A continuação já calculada é reutilizada. Nenhum valor é recalculado aqui.


In [ ]:

def first_existing(cols):
    return next((c for c in cols if c in fine_20_20_df.columns), None)

direction_col = first_existing(["direction", "scan_direction", "sweep_direction"])

if direction_col is None:
    warnings.warn(
        "A tabela carregada não possui coluna de direção. "
        "A comparação forward/backward ficará indisponível, mas a campanha cold-start continua válida."
    )
    continuation_wide = pd.DataFrame({"scenario_uid": cold_summary["scenario_uid"]})
else:
    tmp = fine_20_20_df.copy()
    if "scenario_uid" not in tmp.columns:
        tmp["scenario_uid"] = tmp.apply(stable_uid, axis=1)

    metrics = [c for c in [
        "p_optimal",
        "expected_energy",
        "dominant_bitstring",
        "qgt_numeric_rank",
        "qgt_rank_fraction",
        "qgt_participation_dimension",
        "active_parameter_fraction",
        "dominant_route_replacement_length",
        "dominant_route_stretch",
        "route_target_reached_as_dominant",
        "target_reachability",
    ] if c in tmp.columns]

    keep = ["scenario_uid", direction_col] + metrics
    small = tmp[keep].drop_duplicates(["scenario_uid", direction_col])

    continuation_wide = small.pivot(index="scenario_uid", columns=direction_col, values=metrics)
    continuation_wide.columns = [f"{metric}_{direction}" for metric, direction in continuation_wide.columns]
    continuation_wide = continuation_wide.reset_index()

comparison = cold_summary.merge(continuation_wide, on="scenario_uid", how="left")
display(comparison.head())



## Célula 10 — classificação causal conservadora

A classificação é deliberadamente conservadora.

`ansatz_limit_candidate` **só** é permitido quando existe uma medida independente de `target_reachability`
e ela também é pequena. Sem isso, uma falha persistente fica como `unresolved_failure`.


In [ ]:

def _val(row, names):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            return row[name]
    return np.nan

def classify_row(row):
    cold_success = _val(row, ["cold_success_fraction"])
    p_fw = _val(row, ["p_optimal_forward", "p_optimal_fwd"])
    p_bw = _val(row, ["p_optimal_backward", "p_optimal_bwd"])

    dom_fw = _val(row, ["dominant_bitstring_forward"])
    dom_bw = _val(row, ["dominant_bitstring_backward"])

    reach = _val(row, [
        "target_reachability_forward",
        "target_reachability_backward",
        "target_reachability",
    ])

    path_disagreement = False
    if pd.notna(dom_fw) and pd.notna(dom_bw):
        path_disagreement = str(dom_fw) != str(dom_bw)
    elif pd.notna(p_fw) and pd.notna(p_bw):
        path_disagreement = abs(float(p_fw) - float(p_bw)) > 0.2

    continuation_success = []
    for p in [p_fw, p_bw]:
        if pd.notna(p):
            continuation_success.append(float(p) >= P_OPT_SUCCESS)

    cont_any = any(continuation_success) if continuation_success else np.nan
    cont_all = all(continuation_success) if continuation_success else np.nan

    if pd.notna(cold_success) and cold_success >= 0.8:
        if continuation_success and cont_all and not path_disagreement:
            return "robust"
        if path_disagreement:
            return "multimodal_or_path_dependent"
        if continuation_success and not cont_all:
            return "warm_start_trapping"
        return "cold_robust_continuation_unresolved"

    if pd.notna(cold_success) and cold_success <= 0.2:
        if pd.notna(reach) and float(reach) <= REACHABILITY_TOL:
            return "ansatz_limit_candidate"
        return "unresolved_failure"

    if path_disagreement:
        return "mixed_multimodal"

    return "mixed_or_unresolved"

comparison["causal_regime"] = comparison.apply(classify_row, axis=1)

comparison.to_csv(TABLE_DIR / "cold_vs_continuation_classification_20_21.csv", index=False)

summary_regimes = (
    comparison["causal_regime"]
    .value_counts(dropna=False)
    .rename_axis("causal_regime")
    .reset_index(name="count")
)
display(summary_regimes)



## Célula 11 — gráficos principais

Os gráficos são separados para evitar sobreposição de interpretações:

1. fração de cold starts que recupera o ótimo;
2. melhor \(P_{\rm opt}\) obtido por cold start;
3. diversidade de estados dominantes;
4. geometria QGT, quando disponível.


In [ ]:

for k, g in comparison.groupby("k"):
    g = g.sort_values("delta_from_boundary")

    # 1) taxa de sucesso
    if "cold_success_fraction" in g.columns:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(g["delta_from_boundary"], g["cold_success_fraction"], marker="o")
        ax.axvline(0.0, linestyle="--", linewidth=1)
        ax.set_xlabel(r"$\delta=s-s_c$")
        ax.set_ylabel("fração de cold starts bem-sucedidos")
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"20.21 — recuperação por cold start, k={k}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"cold_success_fraction_k{k}.png", dpi=180)
        plt.show()

    # 2) melhor p_optimal
    if "cold_p_optimal_best" in g.columns:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        y = np.clip(g["cold_p_optimal_best"].astype(float), 1e-16, 1.0)
        ax.semilogy(g["delta_from_boundary"], y, marker="o")
        ax.axvline(0.0, linestyle="--", linewidth=1)
        ax.axhline(P_OPT_SUCCESS, linestyle=":", linewidth=1)
        ax.set_xlabel(r"$\delta=s-s_c$")
        ax.set_ylabel(r"melhor $P_{\rm opt}$ entre cold starts")
        ax.set_title(f"20.21 — melhor recuperação do ótimo, k={k}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"cold_best_poptimal_k{k}.png", dpi=180)
        plt.show()

    # 3) diversidade
    if "cold_unique_dominant_states" in g.columns:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(g["delta_from_boundary"], g["cold_unique_dominant_states"], marker="o")
        ax.axvline(0.0, linestyle="--", linewidth=1)
        ax.set_xlabel(r"$\delta=s-s_c$")
        ax.set_ylabel("nº de estados dominantes distintos")
        ax.set_title(f"20.21 — diversidade de bacias, k={k}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"cold_state_diversity_k{k}.png", dpi=180)
        plt.show()

    # 4) QGT
    if "cold_qgt_rank_fraction_median" in g.columns:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(g["delta_from_boundary"], g["cold_qgt_rank_fraction_median"], marker="o")
        ax.axvline(0.0, linestyle="--", linewidth=1)
        ax.set_xlabel(r"$\delta=s-s_c$")
        ax.set_ylabel(r"mediana $\mathrm{rank}(G)/N_\theta$")
        ax.set_title(f"20.21 — geometria QGT em cold starts, k={k}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"cold_qgt_rank_fraction_k{k}.png", dpi=180)
        plt.show()



## Célula 12 — teste direto de warm-start trapping

Um ponto é especialmente informativo quando:

\[
P_{\rm opt}^{\rm cont}<0.1
\quad\text{e}\quad
\max_j P_{\rm opt}^{\rm cold}(j)>0.9.
\]

Isso mostra que o Hamiltoniano **é solucionável pelo mesmo ansatz/solver**, mas a trajetória de continuação caiu em uma bacia ruim.


In [ ]:

def continuation_best_p(row):
    vals = []
    for c in ["p_optimal_forward", "p_optimal_backward"]:
        if c in row.index and pd.notna(row[c]):
            vals.append(float(row[c]))
    return max(vals) if vals else np.nan

comparison["continuation_best_p_optimal"] = comparison.apply(continuation_best_p, axis=1)

if "cold_p_optimal_best" in comparison.columns:
    comparison["warm_start_trapping_strong"] = (
        (comparison["continuation_best_p_optimal"] < 0.10)
        & (comparison["cold_p_optimal_best"] >= P_OPT_SUCCESS)
    )

    strong_traps = comparison.loc[
        comparison["warm_start_trapping_strong"],
        [c for c in [
            "scenario_uid", "k", "delta_from_boundary", "asset_index",
            "continuation_best_p_optimal", "cold_p_optimal_best",
            "cold_success_fraction", "cold_unique_dominant_states",
        ] if c in comparison.columns]
    ].copy()

    display(strong_traps)
    strong_traps.to_csv(TABLE_DIR / "strong_warm_start_traps_20_21.csv", index=False)
else:
    print("p_optimal não está disponível; teste forte de warm-start trapping não pode ser calculado.")



## Célula 13 — relatório automático e regra de parada

A física só será declarada como reorganização do ansatz se:
- a mudança for reprodutível em cold starts;
- aparecer em observáveis quânticos (QGT/fidelidade/participação);
- não depender apenas da direção da varredura;
- e for separável da troca combinatória clássica.

Caso contrário, o resultado é classificado como landscape/otimizador ou permanece não resolvido.


In [ ]:

report = {
    "experiment": "20.21 cold-start vs continuation",
    "seed": SEED,
    "n_cold_starts": N_COLD_STARTS,
    "n_physical_points": int(len(physical_points_df)),
    "n_cold_runs": int(len(cold_runs)),
    "thresholds": {
        "p_opt_success": P_OPT_SUCCESS,
        "p_opt_strong": P_OPT_STRONG,
        "energy_tol": ENERGY_TOL,
        "reachability_tol": REACHABILITY_TOL,
    },
    "regime_counts": {
        str(r["causal_regime"]): int(r["count"])
        for _, r in summary_regimes.iterrows()
    },
    "interpretation_rule": (
        "all-protocol failure is not called ansatz limitation unless an independent "
        "reachability/expressivity diagnostic also fails"
    ),
}

with open(TABLE_DIR / "experiment_20_21_manifest.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps(report, indent=2, ensure_ascii=False))

print("\nArquivos principais:")
for p in [
    TABLE_DIR / "cold_start_summary_20_21.csv",
    TABLE_DIR / "cold_vs_continuation_classification_20_21.csv",
    TABLE_DIR / "experiment_20_21_manifest.json",
]:
    print(" -", p)



# Como interpretar a 20.21

A ordem de interpretação deve ser:

\[
\boxed{
\text{fronteira clássica}
\rightarrow
\text{dependência de trajetória}
\rightarrow
\text{cold starts}
\rightarrow
\text{geometria/reachability}
}
\]

Não inverter essa ordem.

### Resultado que permite avançar ao Transformer
O dataset pode ser congelado quando:
1. os casos de dependência de caminho forem identificados;
2. os casos recuperáveis por cold start forem separados das falhas persistentes;
3. `asset_tokens` e `block_tokens` continuarem contendo **somente informação pré-solução**;
4. `targets` mantiverem ótimo, energia, QGT, rota, resposta à fronteira e classificação causal apenas como labels.

A 20.21 não tenta “provar transição quântica”. Ela remove um confundidor essencial: **memória do otimizador**.
